# Portfolio API — Angel One (`slug: angelone`)

Exercises all `/portfolio/*` endpoints scoped to the `angelone` source.
Equity holdings via CDP browser fetch (SmartAPI's free tier proved unreliable
for personal sync, so we attach to the authenticated web app instead — same
pattern as Groww and Wint Wealth).

**Auth pre-req:** Log in to [angelone.in](https://angelone.in) inside the AlphaForge Anton Chrome session (`--remote-debugging-port=9299`). Set `ANGELONE_CLIENT_ID` in `backend/.env.cred.local`.

In [ ]:
import json, os
from pathlib import Path

SLUG     = "angelone"
MODE     = "http"          # "in_process" | "http"
BASE     = "http://localhost:8000/api/v1"
FIXTURES = Path.cwd().parent / "tests" / "fixtures" / "broker_csvs"

# Dev defaults from backend/app/core/config.py — override via env if you've
# changed AlphaForge Anton admin credentials.
AF_USERNAME = os.getenv("AF_USERNAME", "admin")
AF_PASSWORD = os.getenv("AF_PASSWORD", "alphaforge-anton-dev")

if MODE == "in_process":
    from fastapi.testclient import TestClient
    from app.main import app
    client = TestClient(app)
    PREFIX = "/api/v1"
else:
    import httpx
    client = httpx.Client(base_url=BASE, timeout=60.0)
    PREFIX = ""


def _login() -> str:
    r = client.post(
        f"{PREFIX}/auth/token",
        data={"username": AF_USERNAME, "password": AF_PASSWORD},
    )
    if r.status_code != 200:
        raise RuntimeError(
            f"Auth failed ({r.status_code}): {r.text}. "
            "Set AF_USERNAME / AF_PASSWORD env vars if you changed admin creds."
        )
    return r.json()["access_token"]


def _ensure_auth() -> None:
    if "Authorization" not in client.headers:
        client.headers["Authorization"] = f"Bearer {_login()}"


def _request(method: str, path: str, **kw):
    _ensure_auth()
    r = client.request(method, f"{PREFIX}{path}", **kw)
    # Self-heal on token expiry / fresh-kernel state.
    if r.status_code == 401:
        client.headers["Authorization"] = f"Bearer {_login()}"
        r = client.request(method, f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text


def get(path, **kw):  return _request("GET", path, **kw)
def post(path, **kw): return _request("POST", path, **kw)

def pp(obj):
    print(json.dumps(obj, indent=2, default=str))

_ensure_auth()
print(f"Mode: {MODE}  slug: {SLUG}  authed as: {AF_USERNAME}")

## 1. Source info

`status: ready` when `ANGELONE_CLIENT_ID` is set, `unconfigured` otherwise.
CSV upload works regardless of status.

In [ ]:
status, body = get(f"/portfolio/sources/{SLUG}")
print(status)
pp(body)

## 2. Sync

Attaches to Chrome via CDP, navigates to `trade.angelone.in/portfolio/holdings`,
reloads the page, and intercepts the holdings XHR. Result is cached to
`~/.alphaforge-anton/portfolio-dumps/angelone-holdings-live.csv`.

> Requires `MODE="http"` with a live server and an open Chrome session
> where you are already logged in to angelone.in.

In [ ]:
status, body = post(f"/portfolio/sources/{SLUG}/sync")
print(status, f"  holdings={body.get('holdings_count')}  status={body.get('info', {}).get('status')}")
for h in (body.get("holdings") or [])[:5]:
    print(f"  {h['symbol']:14}  qty={h['quantity']:<6}  avg=₹{h['avg_price']:>10,.2f}  ltp=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 3. Upload CSV (offline fallback)

Export from `angelone.in` → Portfolio → Holdings → Download CSV. Drop the file at
`tests/fixtures/broker_csvs/angelone_holdings.csv` and re-run this cell.

In [ ]:
csv_path = FIXTURES / "angelone_holdings.csv"
if csv_path.exists():
    with csv_path.open("rb") as f:
        r = client.post(
            f"{PREFIX}/portfolio/sources/{SLUG}/upload",
            files={"file": (csv_path.name, f, "text/csv")},
        )
    print(r.status_code)
    body = r.json()
    print(f"Uploaded {body.get('holdings_count')} holdings")
    for h in (body.get("holdings") or [])[:5]:
        print(f"  {h['symbol']:14}  qty={h['quantity']}")
else:
    print(f"No fixture at {csv_path} — drop an Angel One CSV export there.")

## 4. Holdings — angelone only

Equity holdings only (the holdings page does not expose MF).

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print(status, "  totals:", body.get("totals"))
print(f"\n{len(body.get('holdings', []))} holdings:")
for h in body.get("holdings", []):
    print(f"  {h['symbol']:14}  qty={h['quantity']:<6}  avg=₹{h['avg_price']:>10,.2f}  ltp=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 5. Allocation (angelone)

Expected: `equity`-only allocation.

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print("Allocation:")
for a in body.get("allocation", []):
    print(f"  {a['asset_class']:12} ₹{a['value']:>14,.0f}  ({a['pct']:>5.1f}%)")

## 6. Treemap (angelone)

In [ ]:
status, body = get("/portfolio/treemap", params={"source": SLUG})
print(status)
for c in (body.get("cells") or [])[:10]:
    print(f"  {c['symbol']:14} {c['pct']:>5.1f}% @ ({c['left_pct']:>5.1f}, {c['top_pct']:>5.1f}) {c['width_pct']:>5.1f}x{c['height_pct']:>5.1f}")

## 7. Rebalance (angelone)

In [ ]:
status, body = get("/portfolio/rebalance", params={"source": SLUG})
print("Drift:")
for d in body.get("drift", []):
    print(f"  {d['asset_class']:12} target {d['target_pct']:>5.1f}% · actual {d['actual_pct']:>5.1f}% · drift {d['drift_pct']:>+5.1f}%")
print("\nSuggestions:")
for s in body.get("suggestions", []):
    print("  -", s["action"])

## 8. Free cash

`GET /portfolio/cash` — cached snapshot, always instant.  
`POST /portfolio/cash/{slug}/sync` — opens `trade.angelone.in/funds/funds` via CDP and reads the available-cash XHR (~15–25 s). Requires an active Chrome CDP session.

In [ ]:
# Cached snapshot — instant, no CDP round-trip
_, snap = get("/portfolio/cash")
entry = next((c for c in snap.get("cash", []) if c["source"] == SLUG), None)
if entry:
    avail = "✓" if entry["cash_available"] else "✗ (not yet synced)"
    print(f"Cached  [{avail}] ₹{entry.get('cash', 0):,.2f}  as_of={entry.get('cash_as_of') or 'never'}")
else:
    print("Source not found in /cash response")

# Live sync via CDP
print("\nSyncing …")
status, body = post(f"/portfolio/cash/{SLUG}/sync")
print(f"Status: {status}")
if status == 200:
    c = body["cash"]
    print(f"Fresh   [✓] ₹{c.get('cash', 0):>12,.2f}  as_of={c.get('cash_as_of')}")
else:
    pp(body)

## 9. Standalone dump (bypass FastAPI)

Directly runs the CDP fetch + CSV write without starting the server.
Useful for testing auth and CSV output end-to-end.

In [ ]:
import asyncio, sys
sys.path.insert(0, str(Path.cwd().parent))  # add backend/ to path

from app.modules.brokers.angelone.angelone_dump import dump_angelone

path = await dump_angelone()
print(f"Dumped → {path}")

## 10. Reset angelone cache

In [ ]:
if MODE == "in_process":
    from app.modules.brokers.registry import SOURCES
    SOURCES[SLUG].reset()
    status, body = get(f"/portfolio/sources/{SLUG}")
    print(f"{SLUG}: status={body['status']}  holdings={body['holdings_count']}")
else:
    print("Switch MODE to 'in_process' to reset the in-memory cache directly.")
    print("Or restart the server to clear all cached holdings.")